# RAG 评测（GroUSE）：面向“带引用的 Grounded QA”的指标与评测器

GroUSE（Grounded QA Evaluator）是一套专门评测 **grounded question answering** 的框架：

- 输入：问题（`input`）、模型回答（`actual_output`）、标准答案（`expected_output`，可选）、以及参考上下文（`references`，也就是检索到的证据）
- 输出：一组面向 grounded QA 的指标（如 Answer Relevancy、Completeness、Faithfulness、Usefulness），并带解释

这类评测特别适合“**回答需要按证据标注引用 [i]**”的 RAG 场景：它不只看回答对不对，还会检查回答是否**紧扣问题、是否覆盖关键点、是否忠于证据**，以及在证据不足时回答是否“有用且不胡编”。

<div style="text-align: center;">

<img src="../images/grouse.svg" alt="grouse" style="width:100%; height:auto;">
</div>


## 0) 环境准备

本 notebook 不包含 `pip install`。请确保：

- 已设置 `OPENAI_API_KEY`（建议放在 `../.env`）
- 已安装 `grouse`（GroUSE）与 `nest_asyncio`


In [6]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv("../.env")
sys.path.insert(0, str(Path("..").resolve()))

import nest_asyncio
from grouse import EvaluationSample, GroundedQAEvaluator, meta_evaluate_pipeline

In [7]:
os.environ["HTTP_PROXY"] = os.getenv("HTTP_PROXY", "http://127.0.0.1:7890")
os.environ["HTTPS_PROXY"] = os.getenv("HTTPS_PROXY", "http://127.0.0.1:7890")
os.environ["ALL_PROXY"] = os.getenv("ALL_PROXY", "http://127.0.0.1:7890")

## 1) 在 notebook 里避免嵌套事件循环（可选）

GroUSE 的评测流程内部可能会触发异步调用；在某些 notebook 环境中会遇到 “already running event loop”。这里用 `nest_asyncio` 做兼容处理。


In [8]:
nest_asyncio.apply()

## 2) 初始化评测器

`GroundedQAEvaluator` 会对 grounded QA 的多个维度打分。

你会看到几种典型输出：

- **Answer Relevancy（1-5 或 null）**：回答是否在“回答这个问题”
- **Completeness（1-5 或 null）**：回答是否覆盖了证据里与问题相关的关键信息
- **Faithfulness（0/1 或 null）**：回答是否遵循“每句话有引用且引用正确”的 grounded 约束
- **Usefulness（0/1 或 null）**：当证据不足以回答时，补充信息是否仍然有用且相关


In [ ]:
evaluator = GroundedQAEvaluator(model_name="gpt-4o-mini")

## 3) 评测一个“回答良好”的样本

下面这个样本满足 grounded QA 的几个关键点：

- 回答与问题匹配（relevancy 高）
- 回答覆盖了关键事实（completeness 高）
- 回答带引用，且引用能在 references 里找到对应证据（faithfulness=1）


In [10]:
good_sample = EvaluationSample(
    input="Where is the Eiffel Tower located?",
    actual_output="The Eiffel Tower stands in the Champs de Mars in Paris.[1]",
    expected_output="In the Champs de Mars in Paris. [1]",
    references=[
        "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France"
    ],
)

result = evaluator.evaluate(eval_samples=[good_sample]).evaluations[0]

print("Answer Relevancy (1 to 5):", result.answer_relevancy.answer_relevancy)
print(
    "Answer Relevancy justification:",
    result.answer_relevancy.answer_relevancy_justification,
)
print("Completeness (1 to 5):", result.completeness.completeness)
print("Completeness justification:", result.completeness.completeness_justification)
print("Faithfulness (0 or 1):", result.faithfulness.faithfulness)
print("Faithfulness justification:", result.faithfulness.faithfulness_justification)

100%|██████████| 1/1 [00:10<00:00, 10.04s/it]

2026-06-19 20:46:44,536 - LLM Call Tracker - INFO - Cost: 0.0006$
2026-06-19 20:46:44,536 - LLM Call Tracker - INFO - Cost: 0.0006$
Answer Relevancy (1 to 5): 5
Answer Relevancy justification: This answer also accurately states the location of the Eiffel Tower, effectively responding to the user's question.
Completeness (1 to 5): 5
Completeness justification: Answer 2 also states the location of the Eiffel Tower as the Champs de Mars in Paris and correctly cites the reference. It includes all relevant information from the source.
Faithfulness (0 or 1): 1
Faithfulness justification: The sentence accurately reflects the information from the reference about the Eiffel Tower's location and correctly cites the source.



/home/miao/anaconda3/envs/tp-rag/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/miao/anaconda3/envs/tp-rag/lib/python3.12/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


## 4) 不相关回答：relevancy 会显著下降

回答里虽然是“关于埃菲尔铁塔的事实”，但没有回答“它在哪里”。这属于 **off-topic / mismatch**。

In [11]:
irrelevant_sample = EvaluationSample(
    input="Where is the Eiffel Tower located?",
    actual_output="The Eiffel Tower is mainly made of puddle iron.[2]",
    expected_output="In the Champs de Mars in Paris.[1]",
    references=[
        "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France",
        "The puddle iron (wrought iron) of the Eiffel Tower weighs 7,300 tonnes,[70] and the addition of lifts, shops and antennae have brought the total weight to approximately 10,100 tonnes.",
    ],
)

result = evaluator.evaluate(eval_samples=[irrelevant_sample]).evaluations[0]

print("Answer Relevancy (1 to 5):", result.answer_relevancy.answer_relevancy)
print("Justification:", result.answer_relevancy.answer_relevancy_justification)

100%|██████████| 1/1 [00:08<00:00,  8.80s/it]

2026-06-19 20:46:53,371 - LLM Call Tracker - INFO - Cost: 0.0013$
2026-06-19 20:46:53,371 - LLM Call Tracker - INFO - Cost: 0.0013$
Answer Relevancy (1 to 5): 1
Justification: While the answer provides information about the material of the Eiffel Tower, it does not address the user's question regarding its location.


## 5) 不完整回答：completeness 会下降

这里回答提到了“有人批评 + 一些艺术家”，但漏掉了 references 里更关键、更具体的信息（例如反对的组织、领头人物等）。

In [12]:
incomplete_sample = EvaluationSample(
    input="Who critized the Eiffel Tower project in 1889?",
    actual_output=(
        "The tower was critized by those who did not believe it was feasible and some artists.[1]"
    ),
    expected_output=(
        "The tower was critized by those who did not believe it was feasible and those who objected on artistic grounds.[1]"
        "An artist committee was created to protest againt the construction of the tower, led by the prominent architect "
        "Charles Garnier and including some of the most important figures of the arts, "
        "such as William-Adolphe Bouguereau, Guy de Maupassant, Charles Gounod and Jules Massenet. [2]"
    ),
    references=[
        "The proposed tower had been a subject of controversy, drawing criticism from those who did not believe it was feasible and those who objected on artistic grounds.",
        (
            "It came to a head as work began at the Champ de Mars: a \"Committee of Three Hundred\" "
            "(one member for each metre of the tower's height) was formed, led by the prominent architect "
            "Charles Garnier and including some of the most important figures of the arts, "
            "such as William-Adolphe Bouguereau, Guy de Maupassant, Charles Gounod and Jules Massenet."
        ),
        "A petition called \"Artists against the Eiffel Tower\" was sent to the Minister of Works and Commissioner for the Exposition, Adolphe Alphand, and it was published by Le Temps on 14 February 1887",
    ],
)

result = evaluator.evaluate(eval_samples=[incomplete_sample]).evaluations[0]

print("Completeness (1 to 5):", result.completeness.completeness)
print("Justification:", result.completeness.completeness_justification)

100%|██████████| 1/1 [00:11<00:00, 11.17s/it]

2026-06-19 20:47:04,579 - LLM Call Tracker - INFO - Cost: 0.0021$
2026-06-19 20:47:04,579 - LLM Call Tracker - INFO - Cost: 0.0021$
Completeness (1 to 5): 2
Justification: Answer 2 provides a minimal response, mentioning only that some artists criticized the tower without specifying the committee or key figures involved. It lacks the depth and detail found in the references, particularly missing the mention of the committee and its leader.


## 6) 不忠实回答：faithfulness 会判 0

注意 GroUSE 的 faithfulness 更像是一种“**严格 grounded 约束**”：

- 每句话后面都要有引用 `[i]`
- 引用要能在 references 里找到对应证据
- 不能把证据里的事实“改写成不同含义”


In [13]:
unfaithful_sample = EvaluationSample(
    input="Where is the Eiffel Tower located?",
    actual_output="The Eiffel Tower is located at Rue Rabelais in Paris.[1][2]",
    expected_output="In the Champs de Mars in Paris.[1]",
    references=[
        "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France",
        "Gustave Eiffel died in his appartment at Rue Rabelais in Paris.",
    ],
)

result = evaluator.evaluate(eval_samples=[unfaithful_sample]).evaluations[0]

print("Faithfulness (0 or 1):", result.faithfulness.faithfulness)
print("Justification:", result.faithfulness.faithfulness_justification)

100%|██████████| 1/1 [00:10<00:00, 10.08s/it]

2026-06-19 20:47:14,687 - LLM Call Tracker - INFO - Cost: 0.0027$
2026-06-19 20:47:14,687 - LLM Call Tracker - INFO - Cost: 0.0027$
Faithfulness (0 or 1): 0
Justification: The sentence incorrectly states the location of the Eiffel Tower, which is not at Rue Rabelais but in the Champs de Mars, and it cites the wrong reference.


## 7) “找不到答案”时的 usefulness

当 references 里没有足够信息直接回答问题时，一个好的 grounded 系统应该：

- 明确说明“没有证据能精准回答”
- 但如果 references 里有**相关的周边信息**，可以附带这些信息（仍要引用来源）

GroUSE 用 **Usefulness** 来评估这类“补充信息是否有价值”。


In [14]:
useful_sample = EvaluationSample(
    input="Who critized the Eiffel Tower project in 1889?",
    actual_output=(
        "No document seems to precisely answer your question."
        "However, it is mentioned that a petition against tht Eiffel Tower construciton was sent "
        "to the Minister of Works and Commissioner for the Exposition [1]"
    ),
    expected_output=(
        "No document seems to precisely answer your question."
        "However, it is worth noting that a petition against tht Eiffel Tower construciton was sent "
        "to the Minister of Works and Commissioner for the Exposition [1]"
    ),
    references=[
        "A petition against the tower was sent to the Minister of Works and Commissioner for the Exposition, Adolphe Alphand, and it was published by Le Temps on 14 February 1887"
    ],
)

result = evaluator.evaluate(eval_samples=[useful_sample]).evaluations[0]

print("Usefulness (0 or 1):", result.usefulness.usefulness)
print("Justification:", result.usefulness.usefulness_justification)
print("Positive Acceptance:", result.positive_acceptance)
print("Negative Rejection:", result.negative_rejection)

100%|██████████| 1/1 [00:13<00:00, 13.99s/it]

2026-06-19 20:47:28,709 - LLM Call Tracker - INFO - Cost: 0.0037$
2026-06-19 20:47:28,709 - LLM Call Tracker - INFO - Cost: 0.0037$
Usefulness (0 or 1): 1
Justification: The mention of the petition against the Eiffel Tower construction is relevant and provides useful context regarding the criticism of the project.
Positive Acceptance: 0
Negative Rejection: None


## 8) 批量评测 + 汇总统计

GroUSE 会缓存中间结果，并提供整体统计（例如平均 relevancy / completeness / faithfulness / usefulness）。


In [15]:
evaluation_report = evaluator.evaluate(
    eval_samples=[
        good_sample,
        irrelevant_sample,
        incomplete_sample,
        unfaithful_sample,
        useful_sample,
    ]
).report

print("Average answer relevancy:", evaluation_report.answer_relevancy)
print("Average completeness:", evaluation_report.completeness)
print("Average faithfulness:", evaluation_report.faithfulness)
print("Average usefulness:", evaluation_report.usefulness)

100%|██████████| 5/5 [00:00<00:00, 82.16it/s]

2026-06-19 20:47:28,803 - LLM Call Tracker - INFO - Cost: 0.0074$
2026-06-19 20:47:28,803 - LLM Call Tracker - INFO - Cost: 0.0074$
Average answer relevancy: 3.5
Average completeness: 2.4
Average faithfulness: 0.5
Average usefulness: 1.0


## 9) 用更便宜的模型当 Judge：为什么要“对齐/改写评测 prompt”？

原版 GroUSE 的默认 Judge 质量很好，但成本也更高。一个常见策略是：

- 换一个更便宜的 judge 模型（例如 `gpt-4o-mini`）
- 但为了让它“按同样规则打分”，需要把评测 prompt 适配/迭代

这和 LangChain/LangSmith 文档里提到的 **LLM-as-judge** 思路一致：

- 你写规则（prompt），让 judge 评分
- 然后用人工标注或单元测试集去校准 prompt（迭代对齐）

相关参考（LangSmith 文档）：
- `https://docs.langchain.com/langsmith/evaluation-approaches`（RAG evaluation summary）
- `https://docs.langchain.com/langsmith/llm-as-judge-sdk`（How to define an LLM-as-a-judge evaluator）


### 9.1 Answer Relevancy 的评测 prompt

下面的 prompt 定义了：如何给 **Answer Relevancy** 打分（1-5 或 `null`），以及输出必须满足的 JSON 结构。


In [16]:
relevancy_evaluation_prompt = """# Task

Task: Grounded Question Answering
Based solely on the content of the references, the objective is to generate a response to the user's query. Each statement must be followed by the reference of the source passage, in the format [i] where i is the number of the reference. If no passage seems relevant, the answer should begin with "No document seems to precisely answer your question" and may be supplemented with related sourced information.

# Instructions

I will provide you with two answers, numbered 1 and 2, each containing a response to the user request.
I want you to assign to each answer a relevancy grade between 1 and 5:
- Answer relevancy evaluates if the content of the answer accurately responds to the user's question.
- The truthfulness of the information in the answer does not impact relevancy: even if information that appears false is contained in the answer, as long as this information is related to the request, then relevancy should not decrease. Remember that this information could come from references mentioning imaginary content that you are unaware of: the only thing to evaluate to assign the relevancy grade is therefore the adequacy between the information in the answer and the request, NOT their truthfulness.
- The absence of information in the answer does not impact relevancy, only the information contained in the answer is evaluated.
- Answer relevancy cannot be evaluated if the answer mentions that no document responds to the user request, it is then `null`, regardless of whether it contains other information or not.

Rating scale:
null - The answer asserts that no document precisely responds to the user request. Even if it provides additional information, whether appropriate or not, the relevancy remains `null`.
5 - The answer has excellent relevancy. All information provided in the answer is in line with the question and precisely answers the user request.
4 - The answer achieves good relevancy by providing relevant information to answer the user question. Some information indicated does not exactly answer the question, but remains in line with the request.
3 - The answer has average relevancy, it contains information that allows responding to the user request, but it also contains superfluous information, which was not necessary to answer the request.
2 - The answer shows low relevancy, with some elements related to the request, but the majority of the content is not in line with the question asked.
1 - The answer has very low relevancy, not answering the user's question at all. The content is largely inappropriate or off-topic, delivering no useful information for the request.

Before assigning each grade, you will check that the answer does not contain "No document responds...", if this is the case you must put a grade of `null`. If this is not the case, you will then analyze the adequacy between the request and the information contained in the answer.
Your response should be in JSON format, respecting the following format:
{
    "answer_1": {
        "answer_affirms_no_document_answers": X,
        "answer_relevancy_justification": "...",
        "answer_relevancy": Y
    },
    "answer_2": {
        "answer_affirms_no_document_answers": X,
        "answer_relevancy_justification": "...",
        "answer_relevancy": Y
    }
}
Where "..." is a string, X is a boolean, and Y is an integer between 1 and 5 or `null`.

# Sample

User request: {{ input }}

# To evaluate

Answer 1: {{ expected_output }}
Answer 2: {{ actual_output }}"""


### 9.2 Completeness 的评测 prompt

Completeness 更关注“证据里与问题相关的信息，你的回答有没有覆盖”。


In [17]:
completeness_evaluation_prompt = """# Task

Task: Grounded Question Answering
Based solely on the content of the references, the objective is to generate a response to the user's query. Each statement must be followed by the reference of the source passage, in the format [i] where i is the number of the reference. If no passage seems relevant, the answer should begin with "No document seems to precisely answer your question" and may be supplemented with related sourced information.

# Instructions

I will provide you with two answers, numbered 1 and 2, each containing a response to the user request.
I want you to assign to each answer a completeness grade between 1 and 5:
- The only condition for an answer to be complete is the presence in it of at least all the information from the references that are relevant to the question asked.
- The presence of unrelated information in the answer does not impact completeness.
- The presence of information in the answer not from the references does not impact completeness.
- Possible errors in the sources citing the references do not impact completeness.
- Completeness cannot be evaluated if the references contain no information that can precisely answer the user request, in which case the grade takes the value `null`.

Rating scale:
null - The references contained no relevant information to precisely answer the user's question. In this case, there is no need to read the content of the answer to know that the grade is `null`.
5 - The answer is very complete, it contains all the relevant information from the references. No essential information is omitted, ensuring complete coverage of the question asked.
4 - The answer covers most of the relevant information in depth. It integrates the references satisfactorily, covering the majority of key points. Some details may be missing, but overall, the answer is substantial.
3 - The answer reasonably addresses a number of relevant aspects. It integrates part of the necessary information from the references. However, gaps remain, impacting the overall completeness.
2 - The answer only covers a minimal part of the relevant information. It misses several important information from the references.
1 - The answer covers none of the relevant information, all relevant information from the references has been omitted in the answer.

Before assigning each grade, you will always start by analyzing the information found in the references that are relevant to the user request. If there is no relevant information in the references, completeness must be `null`. If there are relevant information in the references, you will analyze which portion of this information is present or absent in the answers to evaluate the completeness grade. Your response should be in JSON format, respecting the following format:
{
    "answer_1": {
        "completeness_justification": "...",
        "completeness": X
    },
    "answer_2": {
        "completeness_justification": "...",
        "completeness": X
    }
}
Where "..." is a string, and X is an integer between 1 and 5 or `null`.

# SAMPLE

List of references :
{%- for context in contexts %}
Reference {{ loop.index }}: {{ context }}
{%- endfor %}
User request: {{ input }}

# To evaluate

Answer 1: {{ expected_output }}
Answer 2: {{ actual_output }}"""


### 9.3 Faithfulness 的评测 prompt

Faithfulness 在 GroUSE 里是一个更“硬约束”的 grounded 检查：**句子级引用 + 引用正确性**。


In [18]:
faithfulness_evaluation_prompt = """# Task

Task: Grounded Question Answering
Based solely on the content of the references, the objective is to generate a response to the user's query. Each statement must be followed by the reference of the source passage, in the format [i] where i is the number of the reference. If no passage seems relevant, the answer should begin with "No document seems to precisely answer your question" and may be supplemented with related sourced information.

# Instructions

I will provide you with two answers, numbered 1 and 2, each containing a response to the user request.
I want you to assign to each answer a boolean faithfulness grade. An answer is faithful if:
- Each statement made by the answer is followed by a source indicating the reference from which it is drawn.
- The information preceding the source is indeed from the corresponding reference.
- The information preceding the source is in agreement with the corresponding reference, and does not assert facts different from those indicated in the reference.
In all other cases, the response is considered non-faithful.
Faithfulness is also considered non-measurable if the answer asserts that no document responds to the question, and it does not provide any related information, it is then `null`.

Rating scale:
null - The answer asserts that no document responds to the question, and does not provide any related information.
1 - All sentences in the answer cite their sources, and are in agreement with the cited sources.
0 - At least one sentence in the response does not cite its sources, or cites a wrong source, or modifies the content from the references, or asserts something that is not supported by the cited references.

Before assigning each grade, you will start by verifying that the answer does not only assert "No document responds...", without any other information. If this is the case, then faithfulness must be `null`. Otherwise, I want you to analyze by explaining for each sentence, one after the other, if 1) a reference follows the sentence, 2) the reference following the sentence is correct, and 3) if the sentence does not distort or modify the content of the references. Your response should be in JSON format, respecting the following format:
{
    "answer_1": {
        "answer_only_asserts_no_document_answers": X,
        "content_analysis_sentence_by_sentence": [
            {
                "sentence": "...",
                "criterion_1": "...",
                "criterion_2": "...",
                "criterion_3": "..."
            },
            ...
        ],
        "faithfulness_justification": "...",
        "faithfulness": Y
    },
    "answer_2": {
        "answer_only_asserts_no_document_answers": X,
        "content_analysis_sentence_by_sentence": [
            {
            "sentence": "...",
            "criterion_1": "...",
            "criterion_2": "...",
            "criterion_3": "..."
            },
            ...
        ],
        "faithfulness_justification": "...",
        "faithfulness": Y
    }
}
Where "..." is a string, X is a boolean, and Y is either a boolean or `null`.

# Sample

List of references :
{%- for context in contexts %}
Reference {{ loop.index }}: {{ context }}
{%- endfor %}

# To evaluate

Answer 1: {{ expected_output }}
Answer 2: {{ actual_output }}"""


### 9.4 Usefulness 的评测 prompt

Usefulness 只在“回答明确表示没有证据能直接回答”时才有意义：此时评估补充信息是否相关且有价值。


In [19]:
usefulness_evaluation_prompt = """# Task

Task: Grounded Question Answering
Based solely on the content of the references, the objective is to generate a response to the user's query. Each statement must be followed by the reference of the source passage, in the format [i] where i is the number of the reference. If no passage seems relevant, the answer should begin with "No document seems to precisely answer your question" and may be supplemented with related sourced information.

# Instructions

I will provide you with two answers, numbered 1 and 2, each containing a response to the user request.
I want you to assign to each answer a usefulness grade of 0 or 1:
- Usefulness is only evaluated when the answer says that no document precisely answers the user's question, but it still provides information related to the question.
- Usefulness measures how interesting the related information is to know for the user, given that there is no answer in the references.
- If the answer responds to the user request, usefulness must be `null`.
- If the answer indicates that no document responds to the user request, without adding other information, usefulness must be `null`.

Rating scale:
null - (The answer responds to the user request) OR (the answer does not answer the user's question AND does not provide any related information).
1 - The related information is generally related to the question and adds value to the general understanding of the topic.
0 - The related information is completely off-topic with respect to the question asked.

Before assigning each grade, you will start by verifying that the answer indeed asserts "No document responds...", then you will check that the answer contains related information in addition to this assertion. If one of these two conditions is `false` then usefulness must be `null`.
If both conditions are indeed true, then you will analyze the usefulness of having added this related information to evaluate the usefulness grade. Your response should be in JSON format, respecting the following format:
{
    "answer_1": {
        "answer_affirms_no_document_answers": X,
        "answer_contains_related_information": X,
        "usefulness_justification": "...",
        "usefulness": Y
    },
    "answer_2": {
        "answer_affirms_no_document_answers": X,
        "answer_contains_related_information": X,
        "usefulness_justification": "...",
        "usefulness": Y
    }
}
Where "..." is a string, X is a boolean, and Y is an integer that is 0 or 1 or `null`.

# Sample

User request: {{ input }}

# To evaluate

Answer 1: {{ expected_output }}
Answer 2: {{ actual_output }}"""

### 9.5 保存 prompts（供 GroUSE 的 meta-evaluation 使用）

GroUSE 的 `meta_evaluate_pipeline` 会读取你提供的 prompt 模板文件，对 Judge 模型做单元测试式的评估。


In [ ]:
PROMPTS_DIR = Path("gpt4o_mini_prompts")
PROMPTS_DIR.mkdir(parents=True, exist_ok=True)

(PROMPTS_DIR / "answer_relevancy.txt.jinja").write_text(relevancy_evaluation_prompt)
(PROMPTS_DIR / "completeness.txt.jinja").write_text(completeness_evaluation_prompt)
(PROMPTS_DIR / "faithfulness.txt.jinja").write_text(faithfulness_evaluation_prompt)
(PROMPTS_DIR / "usefulness.txt.jinja").write_text(usefulness_evaluation_prompt)

PROMPTS_DIR

PosixPath('gpt4o_mini_prompts')

### 9.6 在 GroUSE 单元测试训练集上评估你的 Judge

这一步会跑 GroUSE 的 unit tests（来自公开数据集），输出聚合指标报告。


In [21]:
meta_evaluations = meta_evaluate_pipeline("gpt-4o-mini", str(PROMPTS_DIR), train_set=True)
print("Aggregated metrics")
print(meta_evaluations.report)


Generating train split:   0%|          | 0/16 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/144 [00:00<?, ? examples/s]

100%|██████████| 16/16 [00:22<00:00,  1.38s/it]

2026-06-19 20:47:57,409 - LLM Call Tracker - INFO - Cost: 0.0170$
2026-06-19 20:47:57,409 - LLM Call Tracker - INFO - Cost: 0.0170$
2026-06-19 20:47:57,409 - LLM Call Tracker - INFO - Cost: 0.0170$
Aggregated metrics
answer_relevancy_success=1.0 completeness_success=0.6875 faithfulness_success=0.625 usefulness_success=0.9375 positive_acceptance_success=0.75 negative_rejection_success=0.6875 total=0.78125


### 9.7 在测试集上评估（更接近泛化表现）

当你在训练集上迭代 prompt 得到满意结果后，再用测试集做一次验证。


In [22]:
meta_evaluations = meta_evaluate_pipeline("gpt-4o-mini", str(PROMPTS_DIR), train_set=False)
meta_evaluations.report


100%|██████████| 144/144 [01:54<00:00,  1.26it/s]

2026-06-19 20:49:55,791 - LLM Call Tracker - INFO - Cost: 0.1714$
2026-06-19 20:49:55,791 - LLM Call Tracker - INFO - Cost: 0.1714$
2026-06-19 20:49:55,791 - LLM Call Tracker - INFO - Cost: 0.1714$
2026-06-19 20:49:55,791 - LLM Call Tracker - INFO - Cost: 0.1714$


MetaEvalReport(answer_relevancy_success=0.9236111111111112, completeness_success=0.6458333333333334, faithfulness_success=0.7361111111111112, usefulness_success=0.9375, positive_acceptance_success=0.7847222222222222, negative_rejection_success=0.7222222222222222, total=0.7916666666666666)

## 10) 局限性

- 单元测试/离线评测能帮你发现 judge 的薄弱点，但不等价于“线上绝对可靠”。
- 对 LLM-as-judge 的分数要保留审慎：定期抽样人工复核，才能避免评测本身漂移。

## 参考

```latex
@misc{muller2024grousebenchmarkevaluateevaluators,
      title={GroUSE: A Benchmark to Evaluate Evaluators in Grounded Question Answering}, 
      author={Sacha Muller and António Loison and Bilel Omrani and Gautier Viaud},
      year={2024},
      eprint={2409.06595},
      archivePrefix={arXiv},
      primaryClass={cs.CL},
      url={https://arxiv.org/abs/2409.06595}, 
}
```
